# RENKIN — Retrosynthesis Engine Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kent-tokyo/renkin/blob/master/examples/renkin_quickstart.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-kent--tokyo%2Frenkin-blue?logo=github)](https://github.com/kent-tokyo/renkin)

**RENKIN** is a pure-Rust retrosynthesis engine for Computer-Aided Synthesis Planning (CASP).  
Given a target molecule (SMILES), it finds multi-step synthesis routes back to cheap commercial starting materials.

- **78.0% Top-1** on USPTO-50k (4,907 molecules, depth=5, beam=100)
- Pure Rust · Zero C/C++ dependencies · Python / WASM / CLI
- [Documentation](https://kent-tokyo.github.io/renkin/) · [Playground](https://kent-tokyo.github.io/renkin/playground/)

## 1. Install

In [ ]:
!pip install renkin --quiet

## 2. Basic Usage — Aspirin Retrosynthesis

Find synthesis routes for aspirin (`CC(=O)Oc1ccccc1C(=O)O`).

In [ ]:
import renkin

result = renkin.find_routes(
    "CC(=O)Oc1ccccc1C(=O)O",  # Aspirin
    depth=5,
    max_routes=5,
    beam_width=100,
)

print(f"Found {result['routes_found']} route(s) for: {result['target']}\n")

for i, route in enumerate(result['routes'], 1):
    print(f"--- Route {i} (depth {route['depth']}) ---")
    for step in route['steps']:
        precs = ' + '.join(step['precursors'])
        print(f"  {step['target']}")
        print(f"    → {precs}")
        print(f"       [{step['rule']}]")
    print()

## 3. More Examples

Try different drug-like molecules.

In [ ]:
targets = {
    "Aspirin":          "CC(=O)Oc1ccccc1C(=O)O",
    "Paracetamol":      "CC(=O)Nc1ccc(O)cc1",
    "Biphenyl":         "c1ccc(-c2ccccc2)cc1",
    "4-Phenylpyridine": "c1ccc(-c2ccncc2)cc1",
    "Ibuprofen":        "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
}

for name, smiles in targets.items():
    result = renkin.find_routes(smiles, depth=5, max_routes=1, beam_width=100)
    n = result['routes_found']
    if n > 0:
        depth = result['routes'][0]['depth']
        rule = result['routes'][0]['steps'][0]['rule'] if result['routes'][0]['steps'] else 'BB'
        print(f"✓  {name:20s}  depth={depth}  [{rule}]")
    else:
        print(f"✗  {name:20s}  no route found")

## 4. Try Your Own Molecule

Enter any SMILES string and run retrosynthesis.

In [ ]:
# Change this SMILES to your target molecule
target_smiles = "COc1ccc(CCN)cc1"  # 4-methoxyphenethylamine (mescaline precursor)

result = renkin.find_routes(
    target_smiles,
    depth=5,
    max_routes=3,
    beam_width=100,
)

print(f"Target: {result['target']}")
print(f"Routes found: {result['routes_found']}\n")

for i, route in enumerate(result['routes'], 1):
    print(f"Route {i} (depth {route['depth']}):")
    for step in route['steps']:
        print(f"  {step['target']} → {' + '.join(step['precursors'])}  [{step['rule']}]")
    print()

## 5. Visualize with RDKit (optional)

Draw reaction steps as 2D structures.

In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Draw
    from IPython.display import display, Image
    import io

    target = "CC(=O)Oc1ccccc1C(=O)O"  # Aspirin
    result = renkin.find_routes(target, depth=5, max_routes=1, beam_width=100)

    if result['routes_found'] > 0:
        route = result['routes'][0]
        smiles_list = [target]
        labels = ["Target"]
        for step in route['steps']:
            for prec in step['precursors']:
                smiles_list.append(prec)
                labels.append(step['rule'])

        mols = [Chem.MolFromSmiles(s) for s in smiles_list if Chem.MolFromSmiles(s)]
        img = Draw.MolsToGridImage(
            mols[:8],
            molsPerRow=4,
            subImgSize=(300, 200),
            legends=labels[:8],
        )
        display(img)
    else:
        print("No route found.")

except ImportError:
    print("Install rdkit for visualization: pip install rdkit")
    print("In Colab: !pip install rdkit")

## 6. Python API Reference

```python
renkin.find_routes(
    smiles: str,           # Target molecule SMILES
    depth: int = 5,        # Max retrosynthesis depth
    max_routes: int = 5,   # Max routes to return
    beam_width: int = 0,   # Beam width (0 = unlimited A*)
    building_blocks: list[str] | None = None,  # Custom stock (canonical SMILES)
) -> dict
```

**Return value:**
```python
{
    "target": str,           # Input SMILES (canonicalized)
    "routes_found": int,
    "routes": [
        {
            "depth": int,
            "steps": [
                {
                    "rule": str,           # Reaction rule name
                    "target": str,         # Molecule being disconnected
                    "precursors": [str],   # Resulting fragments (SMILES)
                }
            ]
        }
    ]
}
```

**Links:**  
- [Full documentation](https://kent-tokyo.github.io/renkin/)  
- [GitHub](https://github.com/kent-tokyo/renkin)  
- [crates.io](https://crates.io/crates/renkin) · [PyPI](https://pypi.org/project/renkin/)  
- [Live playground (WASM)](https://kent-tokyo.github.io/renkin/playground/)